In [ ]:
import pandas as pd

# 读取 result.csv（每个 mock object 原始数据）
df = pd.read_csv("result.csv")

# 确保关键列为数值类型
numeric_cols = ["TestCount", "AvgAddedCCTR", "AddedCCTR"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# 按 MockLevel 分组聚合
summary = df.groupby("MockLevel").agg(
    MockObjectCount=("MockID", "count"),
    TotalTestCount=("TestCount", "sum"),
    AvgTestPerMock=("TestCount", "mean"),               # 平均每个 Mock 影响的测试数
    AvgAddedCCTR_perTest=("AvgAddedCCTR", "mean"),      # 平均每个测试增加的复杂度
    AvgAddedCCTR_perMock=("AddedCCTR", "mean")          # 平均每个 Mock 增加的复杂度
).reset_index()

# 输出结果
print(summary)

# 可选：保存到 CSV
summary.to_csv("CCTR_Level_Summary.csv", index=False)


: 

In [ ]:
import pandas as pd
import numpy as np

# =============================
# 读取两个 CSV 文件
# =============================
df_level0 = pd.read_csv("L0_conversion_result.csv")
df_result = pd.read_csv("result.csv")

# -----------------------------
# 1️⃣ Level 0 → Level 1 / 2
# -----------------------------
df_a = pd.DataFrame()
df_a["Project"] = df_level0["Project"]
df_a["ConvertedFrom"] = 0
df_a["ConvertedTo"] = df_level0["ConvertedLevel"]
df_a["BeforeCount"] = df_level0["L0Count"]
df_a["AfterCount"] = 1
df_a["CCTRChange"] = -1 * df_level0["CCTRReduction"]
df_a["CCTRChangePerTest"] = -1 * df_level0["CCTRReductionPerTest"]
df_a["AddedCCTR"] = df_level0["AddedCCTR"]
df_a["RawCCTR"] = df_level0["RawCCTR"]

# 防止除零
mask_a = df_a["AddedCCTR"] != 0
df_a["CCTRChangePercent"] = np.where(mask_a, df_a["CCTRChange"] / df_a["AddedCCTR"], np.nan)

# -----------------------------
# 2️⃣ Level 1 / 2 → Level 0
# -----------------------------
df_b = df_result[df_result["MockLevel"].isin([1, 2])].copy()
df_b["ConvertedFrom"] = df_b["MockLevel"]
df_b["ConvertedTo"] = 0
df_b["BeforeCount"] = 1
df_b["AfterCount"] = df_b["TestCount"]
df_b["CCTRChange"] = (df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]) * df_b["TestCount"]
df_b["CCTRChangePerTest"] = df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]
df_b["AddedCCTR"] = df_b["AvgAddedCCTR"]
df_b["RawCCTR"] = df_b["Level0CCTR"]

# ✅ 防止除零
mask_b = df_b["AddedCCTR"] != 0
df_b["CCTRChangePercent"] = np.where(mask_b, df_b["CCTRChange"] / df_b["AddedCCTR"], np.nan)

# -----------------------------
# ✅ 分别汇总 df_a 和 df_b（不分组）
# -----------------------------
summary_a = pd.DataFrame({
    "ConvertedFrom": [0],
    "ConvertedTo": ["1 / 2"],
    "ConversionCount": [len(df_a)],
    "TotalBeforeMocks": [df_a["BeforeCount"].sum()],
    "TotalAfterMocks": [df_a["AfterCount"].sum()],
    "AvgBeforeCount": [df_a["BeforeCount"].mean()],
    "AvgAfterCount": [df_a["AfterCount"].mean()],
    "AvgAddedCCTR": [df_a["AddedCCTR"].mean()],
    "AvgCCTRChange": [df_a["CCTRChange"].mean()],
    "AvgCCTRChangePerTest": [df_a["CCTRChangePerTest"].mean()],
    "AvgCCTRChangePercent": [(df_a["CCTRChangePercent"].mean() * 100).round(2)]
})

summary_b = pd.DataFrame({
    "ConvertedFrom": ["1 / 2"],
    "ConvertedTo": [0],
    "ConversionCount": [len(df_b)],
    "TotalBeforeMocks": [df_b["BeforeCount"].sum()],
    "TotalAfterMocks": [df_b["AfterCount"].sum()],
    "AvgBeforeCount": [df_b["BeforeCount"].mean()],
    "AvgAfterCount": [df_b["AfterCount"].mean()],
    "AvgAddedCCTR": [df_b["AddedCCTR"].mean()],
    "AvgCCTRChange": [df_b["CCTRChange"].mean()],
    "AvgCCTRChangePerTest": [df_b["CCTRChangePerTest"].mean()],
    "AvgCCTRChangePercent": [(df_b["CCTRChangePercent"].mean() * 100).round(2)]
})

# -----------------------------
# 合并输出
# -----------------------------
final_summary = pd.concat([summary_a, summary_b], ignore_index=True)

print(final_summary)
final_summary.to_csv("CCTR_Conversion_Summary_Simple.csv", index=False)


  ConvertedFrom ConvertedTo  ConversionCount  TotalBeforeMocks  \
0             0       1 / 2             7510             34199   
1         1 / 2           0            16399             16399   

   TotalAfterMocks  AvgBeforeCount  AvgAfterCount  AvgAddedCCTR  \
0             7510        4.553795       1.000000     21.167377   
1            56889        1.000000       3.469053      2.594087   

   AvgCCTRChange  AvgCCTRChangePerTest  AvgCCTRChangePercent  
0     -11.700133             -2.628349                -63.69  
1      36.409598              3.244373               1686.22  


In [ ]:
import pandas as pd

# 读取两个 CSV 文件
df_level0 = pd.read_csv("L0_conversion_result.csv")
df_result = pd.read_csv("result.csv")

# -----------------------------
# 1️⃣ Level 0 → Level 1 / 2
# -----------------------------
df_a = pd.DataFrame()
df_a["Project"] = df_level0["Project"]
df_a["ConvertedFrom"] = 0
df_a["ConvertedTo"] = df_level0["ConvertedLevel"]
df_a["BeforeCount"] = df_level0["L0Count"]
df_a["AfterCount"] = 1
df_a["AvgBeforeCount"] = df_level0["L0Count"]
df_a["AvgAfterCount"] = 1
df_a["CCTRChange"] = -1 * df_level0["CCTRReduction"]
df_a["CCTRChangePerTest"] = -1 * df_level0["CCTRReductionPerTest"]
df_a['RawCCTR'] = df_level0['RawCCTR']
df_a["%"] = df_a["CCTRChange"] / df_level0["RawCCTR"]

# -----------------------------
# 2️⃣ Level 1 / 2 → Level 0
# -----------------------------
df_b = df_result[df_result["MockLevel"].isin([1, 2])].copy()

df_b["Project"] = df_b["Project"]
df_b["ConvertedFrom"] = df_b["MockLevel"]
df_b["ConvertedTo"] = 0
df_b["BeforeCount"] = 1
df_b["AfterCount"] = df_b["TestCount"]
df_b["AvgBeforeCount"] = 1
df_b["AvgAfterCount"] = df_b["TestCount"]
df_b["CCTRChange"] = (df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]) * df_b["TestCount"]
df_b["CCTRChangePerTest"] = df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]
df_b['RawCCTR'] = df_result['RawCCTR']
df_b["%"] = df_b["CCTRChange"] / df_result["RawCCTR"]
# -----------------------------
# 只保留统一的列结构
# -----------------------------
cols = [
    "Project",
    "ConvertedFrom",
    "ConvertedTo",
    "BeforeCount",
    "AfterCount",
    "CCTRChange",
    "CCTRChangePerTest",
    "RawCCTR",
    "%"
]

df_a = df_a[cols]
df_b = df_b[cols]

# -----------------------------
# 合并两类结果
# -----------------------------
conversion_summary = pd.concat([df_a, df_b], ignore_index=True)

# 保存到 CSV
conversion_summary.to_csv("CCTR_Conversion_Summary.csv", index=False)

print(conversion_summary.head())

# 按转换类型分组聚合
group_summary = conversion_summary.groupby(["ConvertedFrom", "ConvertedTo"]).agg(
    ConversionCount=("Project", "count"),
    TotalBeforeMocks=("BeforeCount", "sum"),
    TotalAfterMocks=("AfterCount", "sum"),
    AvgBeforeCount=("BeforeCount", "mean"),
    AvgAfterCount=("AfterCount", "mean"),
    AvgCCTRChange=("CCTRChange", "mean"),
    AvgCCTRChangePerTest=("CCTRChangePerTest", "mean"),    
    Prestangeimpact=("%", "mean")
    
)
# 输出或保存
print(group_summary)
group_summary.to_csv("CCTR_Conversion_Group_Summary.csv", index=False)


: 

In [ ]:
import pandas as pd

# =============================
# 读取两个 CSV 文件
# =============================
df_level0 = pd.read_csv("L0_conversion_result.csv")
df_result = pd.read_csv("result.csv")

# -----------------------------
# 1️⃣ Level 0 → Level 1 / 2
# -----------------------------
df_a = pd.DataFrame()
df_a["Project"] = df_level0["Project"]
df_a["ConvertedFrom"] = 0
df_a["ConvertedTo"] = df_level0["ConvertedLevel"]
df_a["BeforeCount"] = df_level0["L0Count"]
df_a["AfterCount"] = 1
df_a["CCTRChange"] = -1 * df_level0["CCTRReduction"]
df_a["CCTRChangePerTest"] = -1 * df_level0["CCTRReductionPerTest"]
df_a["AddedCCTR"] = df_level0["AddedCCTR"]

# 计算比例
df_a["CCTRChangePercent"] = df_a["CCTRChange"] / df_a["AddedCCTR"]

# -----------------------------
# 2️⃣ Level 1 / 2 → Level 0
# -----------------------------
df_b = df_result[df_result["MockLevel"].isin([1, 2])].copy()
df_b["ConvertedFrom"] = df_b["MockLevel"]
df_b["ConvertedTo"] = 0
df_b["BeforeCount"] = 1
df_b["AfterCount"] = df_b["TestCount"]
df_b["CCTRChange"] = (df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]) * df_b["TestCount"]
df_b["CCTRChangePerTest"] = df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]
df_b["AddedCCTR"] = df_b["AvgAddedCCTR"]

# 计算比例
df_b["CCTRChangePercent"] = df_b["CCTRChange"] / df_b["AddedCCTR"]

# -----------------------------
# ✅ 分别汇总 df_a 和 df_b（不分组）
# -----------------------------
summary_a = pd.DataFrame({
    "ConvertedFrom": [0],
    "ConvertedTo": ["1 / 2"],
    "ConversionCount": [len(df_a)],
    "TotalBeforeMocks": [df_a["BeforeCount"].sum()],
    "TotalAfterMocks": [df_a["AfterCount"].sum()],
    "AvgBeforeCount": [df_a["BeforeCount"].mean()],
    "AvgAfterCount": [df_a["AfterCount"].mean()],
    "AvgAddedCCTR": [df_a["AddedCCTR"].mean()],
    "AvgCCTRChange": [df_a["CCTRChange"].mean()],
    "AvgCCTRChangePerTest": [df_a["CCTRChangePerTest"].mean()],
    "AvgCCTRChangePercent": [(df_a["CCTRChangePercent"].mean() * 100).round(2)]
})

summary_b = pd.DataFrame({
    "ConvertedFrom": ["1 / 2"],
    "ConvertedTo": [0],
    "ConversionCount": [len(df_b)],
    "TotalBeforeMocks": [df_b["BeforeCount"].sum()],
    "TotalAfterMocks": [df_b["AfterCount"].sum()],
    "AvgBeforeCount": [df_b["BeforeCount"].mean()],
    "AvgAfterCount": [df_b["AfterCount"].mean()],
    "AvgAddedCCTR": [df_b["AddedCCTR"].mean()],
    "AvgCCTRChange": [df_b["CCTRChange"].mean()],
    "AvgCCTRChangePerTest": [df_b["CCTRChangePerTest"].mean()],
    "AvgCCTRChangePercent": [(df_b["CCTRChangePercent"].mean() * 100).round(2)]
})

# -----------------------------
# 合并输出
# -----------------------------
final_summary = pd.concat([summary_a, summary_b], ignore_index=True)

print(final_summary)
final_summary.to_csv("CCTR_Conversion_Summary_Simple.csv", index=False)


In [12]:
import pandas as pd

# 读取两个 CSV 文件
df_level0 = pd.read_csv("L0_conversion_result.csv")
df_result = pd.read_csv("result.csv")

# -----------------------------
# 1️⃣ Level 0 → Level 1 / 2
# -----------------------------
df_a = pd.DataFrame()
df_a["Project"] = df_level0["Project"]
df_a["ConvertedFrom"] = 0
df_a["ConvertedTo"] = df_level0["ConvertedLevel"]
df_a["BeforeCount"] = df_level0["L0Count"]
df_a["AfterCount"] = 1
df_a["AvgBeforeCount"] = df_level0["L0Count"]
df_a["AvgAfterCount"] = 1
df_a["CCTRChange"] = -1 * df_level0["CCTRReduction"]
df_a["CCTRChangePerTest"] = -1 * df_level0["CCTRReductionPerTest"]

# -----------------------------
# 2️⃣ Level 1 / 2 → Level 0
# -----------------------------
df_b = df_result[df_result["MockLevel"].isin([1, 2])].copy()

df_b["Project"] = df_b["Project"]
df_b["ConvertedFrom"] = df_b["MockLevel"]
df_b["ConvertedTo"] = 0
df_b["BeforeCount"] = 1
df_b["AfterCount"] = df_b["TestCount"]
df_b["AvgBeforeCount"] = 1
df_b["AvgAfterCount"] = df_b["TestCount"]
df_b["CCTRChange"] = (df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]) * df_b["TestCount"]
df_b["CCTRChangePerTest"] = df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]

# -----------------------------
# 只保留统一的列结构
# -----------------------------
cols = [
    "Project",
    "ConvertedFrom",
    "ConvertedTo",
    "BeforeCount",
    "AfterCount",
    "CCTRChange",
    "CCTRChangePerTest"
]

df_a = df_a[cols]
df_b = df_b[cols]

# -----------------------------
# 合并两类结果
# -----------------------------
conversion_summary = pd.concat([df_a, df_b], ignore_index=True)

# 保存到 CSV
conversion_summary.to_csv("CCTR_Conversion_Summary.csv", index=False)

print(conversion_summary.head())

# 按转换类型分组聚合
group_summary = conversion_summary.groupby(["ConvertedFrom", "ConvertedTo"]).agg(
    ConversionCount=("Project", "count"),
    TotalBeforeMocks=("BeforeCount", "sum"),
    TotalAfterMocks=("AfterCount", "sum"),
    AvgBeforeCount=("BeforeCount", "mean"),
    AvgAfterCount=("AfterCount", "mean"),
    AvgCCTRChange=("CCTRChange", "mean"),
    AvgCCTRChangePerTest=("CCTRChangePerTest", "mean")
).reset_index()

# 输出或保存
print(group_summary)
group_summary.to_csv("CCTR_Conversion_Group_Summary.csv", index=False)

    Project  ConvertedFrom  ConvertedTo  BeforeCount  AfterCount  CCTRChange  \
0  ActiveMQ              0            2            2           1        -6.0   
1  ActiveMQ              0            1            2           1         0.0   
2  ActiveMQ              0            2            2           1        -4.0   
3  ActiveMQ              0            1            2           1       -14.0   
4  ActiveMQ              0            1            2           1        -2.0   

   CCTRChangePerTest  
0               -3.0  
1               -0.0  
2               -2.0  
3               -7.0  
4               -1.0  
   ConvertedFrom  ConvertedTo  ConversionCount  TotalBeforeMocks  \
0              0            1             5337             20405   
1              0            2             2173             13794   
2              1            0             9718              9718   
3              2            0             6681              6681   

   TotalAfterMocks  AvgBeforeCount  AvgA

需要你帮我画这样一个图：
读取一个csv文件。格式如下
project_name,rawMockObjectId,mockPatternLevel,avg_added_cctr_per_test_case,Level_0_added_cctr_per_test_case,raw_code_cctr,re_mo_code_cctr,test_case_count,added_cctr
ActiveMQ,1,0,4.0,4.0,18,14,1,4
ActiveMQ,2,0,4.0,4.0,18,14,1,4
ActiveMQ,3,0,5.0,5.0,5,0,1,5
ActiveMQ,4,0,0.0,0.0,13,13,1,0

然后它的文件名为result.csv。
其中统计两件事。1，以

In [4]:
import pandas as pd

# =============================
# 读取两个 CSV 文件
# =============================
df_level0 = pd.read_csv("L0_conversion_result.csv")
df_result = pd.read_csv("result.csv")

# -----------------------------
# 1️⃣ Level 0 → Level 1 / 2
# -----------------------------
df_a = pd.DataFrame()
df_a["Project"] = df_level0["Project"]
df_a["ConvertedFrom"] = 0
df_a["ConvertedTo"] = df_level0["ConvertedLevel"]
df_a["BeforeCount"] = df_level0["L0Count"]
df_a["AfterCount"] = 1
df_a["CCTRChange"] = -1 * df_level0["CCTRReduction"]
df_a["CCTRChangePerTest"] = -1 * df_level0["CCTRReductionPerTest"]
df_a["AddedCCTR"] = df_level0["AddedCCTR"]

# -----------------------------
# 2️⃣ Level 1 / 2 → Level 0
# -----------------------------
df_b = df_result[df_result["MockLevel"].isin([1, 2])].copy()
df_b["ConvertedFrom"] = df_b["MockLevel"]
df_b["ConvertedTo"] = 0
df_b["BeforeCount"] = 1
df_b["AfterCount"] = df_b["TestCount"]
df_b["CCTRChange"] = (df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]) * df_b["TestCount"]
df_b["CCTRChangePerTest"] = df_b["Level0CCTR"] - df_b["AvgAddedCCTR"]
df_b["AddedCCTR"] = df_b["AvgAddedCCTR"]

# -----------------------------
# 合并两类结果
# -----------------------------
cols = [
    "Project",
    "ConvertedFrom",
    "ConvertedTo",
    "BeforeCount",
    "AfterCount",
    "CCTRChange",
    "CCTRChangePerTest",
    "AddedCCTR"
]

conversion_summary = pd.concat([df_a[cols], df_b[cols]], ignore_index=True)

# -----------------------------
# ✅ 新增比例列（逐条求比）
# -----------------------------
conversion_summary["CCTRChangePercent"] = (
    conversion_summary["CCTRChange"] / conversion_summary["AddedCCTR"]
)

# -----------------------------
# 聚合汇总
# -----------------------------
group_summary = conversion_summary.groupby(["ConvertedFrom", "ConvertedTo"]).agg(
    ConversionCount=("Project", "count"),
    TotalBeforeMocks=("BeforeCount", "sum"),
    TotalAfterMocks=("AfterCount", "sum"),
    AvgBeforeCount=("BeforeCount", "mean"),
    AvgAfterCount=("AfterCount", "mean"),
    AvgAddedCCTR=("AddedCCTR", "mean"),
    AvgCCTRChange=("CCTRChange", "mean"),
    AvgCCTRChangePerTest=("CCTRChangePerTest", "mean"),
    AvgCCTRChangePercent=("CCTRChangePercent", "mean")
).reset_index()

# 转换为百分比显示
group_summary["AvgCCTRChangePercent"] = (group_summary["AvgCCTRChangePercent"] * 100).round(2)

# -----------------------------
# 输出
# -----------------------------
print(group_summary)
group_summary.to_csv("CCTR_Conversion_Group_Summary.csv", index=False)


   ConvertedFrom  ConvertedTo  ConversionCount  TotalBeforeMocks  \
0              0            1             5337             20405   
1              0            2             2173             13794   
2              1            0             9718              9718   
3              2            0             6681              6681   

   TotalAfterMocks  AvgBeforeCount  AvgAfterCount  AvgAddedCCTR  \
0             5337        3.823309       1.000000     14.456811   
1             2173        6.347906       1.000000     37.648873   
2            24885        1.000000       2.560712      1.732546   
3            32004        1.000000       4.790301      3.847261   

   AvgCCTRChange  AvgCCTRChangePerTest  AvgCCTRChangePercent  
0      -8.884767             -2.372211                -68.61  
1     -18.614818             -3.257435                -51.90  
2       3.539514              1.066023                   inf  
3      84.221524              6.412941                   inf  


In [ ]:
import os
import subprocess
import xml.etree.ElementTree as ET
import re
import csv
from pathlib import Path
from tree_sitter import Language, Parser
import tree_sitter_java as tsjava
import pandas as pd

# === Tree-sitter Java Initialization ===
JAVA_LANGUAGE = Language(tsjava.language())
parser = Parser(JAVA_LANGUAGE)

# # === Resource Paths ===
# SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

OUTPUT_CSV = os.path.join(r"C:\CCTR\complexity_summary.csv")
# os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)


In [2]:
# === Test-Aware Cognitive Complexity Calculator ===
class CognitiveComplexityCalculatorTestAware:
    def __init__(self, code, method_name=None):
        self.code = code.encode("utf-8")
        self.method_name = method_name
        self.complexity = 0
        self.nesting_level = 0
        self.parser = parser

    def compute_complexity(self):
        tree = self.parser.parse(self.code)
        self._analyze_node(tree.root_node)
        return self.complexity

    def _analyze_node(self, node):
        for child in node.children:
            kind = child.type
            text = self.code[child.start_byte:child.end_byte].decode("utf-8")
            if kind in ["if_statement", "for_statement", "while_statement", "do_statement", "switch_statement", "catch_clause"]:
                self._increment(child)
            elif kind == "binary_expression" and ("&&" in text or "||" in text):
                self.complexity += 1
            elif kind == "labeled_statement" and any(k in text for k in ["break", "continue", "goto"]):
                self.complexity += 1
            elif kind == "method_invocation":
                if self._is_recursive_call(text):
                    self.complexity += 1
                if any(x in text for x in ["mock(", "when(", "verify("]):
                    self.complexity += 1
                if "assert" in text or "fail(" in text:
                    self.complexity += 1
            elif kind == "annotation":
                if "@Test" in text:
                    self.complexity += 1
                elif "@ParameterizedTest" in text:
                    self.complexity += 2
                elif "@BeforeEach" in text or "@AfterEach" in text:
                    self.complexity += 1
            self._analyze_node(child)

    def _increment(self, node):
        self.complexity += 1 + self.nesting_level
        self.nesting_level += 1
        self._analyze_node(node)
        self.nesting_level -= 1

    def _is_recursive_call(self, text):
        return self.method_name and self.method_name in text



In [25]:
import json
import os

def analyze_mock_objects(json_path):
    """
    分析指定 JSON 文件中每个 mock object 对测试复杂度 (CCTR) 的影响。

    输出：
    [
      {
        "project_name": "ActiveMQ",
        "rawMockObjectId": 1,
        "mockPatternLevel": 0,
        "added_cctr": 12.0,
        "avg_added_cctr_per_test_case": 6.0,
        "raw_code_cctr": 34.0,
        "re_mo_code_cctr": 22.0,
        "test_case_count": 2
      },
      ...
    ]
    """
    project_name = os.path.splitext(os.path.basename(json_path))[0]
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    results = []

    for obj in data:
        rawMockObjectId = obj.get("rawMockObjectId")
        mockPatternLevel = obj.get("mockPatternLevel", 0)

        # 收集所有 test 方法代码
        test_case_count = 0
        raw_code = set()
        for statement in obj.get("statements", []):
            locationContext = statement.get("locationContext", {})
            method_code = locationContext.get("methodRawCode", "")
            if not method_code:
                continue
            if method_code not in raw_code:
                raw_code.add(method_code)
                if (statement.get("locate", "") == "Test Case" or
                    "Test" in locationContext.get("methodAnnotations", []) or
                    "test" in locationContext.get("methodName", "").lower()):
                    test_case_count += 1

        if mockPatternLevel == 0:
            test_case_count = 1
        elif test_case_count == 0:
            test_case_count = 2  # 避免除以零
        
        if mockPatternLevel ==1 or mockPatternLevel ==2:
            mockPatternLevel =1
        elif mockPatternLevel >=3:
            mockPatternLevel =2

        # 拼接所有方法代码
        raw_code_text = "\n".join(raw_code)
        re_mo_code_text = raw_code_text

        # 去除当前 mock object 的语句
        for statement in obj.get("statements", []):
            st_code = statement.get("code", "")
            if st_code:
                re_mo_code_text = re_mo_code_text.replace(st_code, "")

        # 计算复杂度
        raw_code_cctr = CognitiveComplexityCalculatorTestAware(raw_code_text).compute_complexity()
        re_mo_code_cctr = CognitiveComplexityCalculatorTestAware(re_mo_code_text).compute_complexity()

        added_cctr = raw_code_cctr - re_mo_code_cctr
        avg_added_cctr = added_cctr / test_case_count if test_case_count > 0 else 0.0

        results.append({
            "project_name": project_name,
            "rawMockObjectId": rawMockObjectId,
            "mockPatternLevel": mockPatternLevel,
            "added_cctr": added_cctr,
            "avg_added_cctr_per_test_case": avg_added_cctr,
            "raw_code_cctr": raw_code_cctr,
            "re_mo_code_cctr": re_mo_code_cctr,
            "test_case_count": test_case_count
        })

    return results


In [26]:
data=analyze_mock_objects(r"C:\CCTR\mock object\CloudStack.json")
df = pd.DataFrame(data)
df.to_csv(r'C:\CCTR\result.csv', index=False)


In [ ]:
from glob import glob

# 获取所有 mock object 目录下的 json 文件
json_files = glob(r"C:\CCTR\mock object\*.json")

all_data = []
for json_path in json_files:
    result = analyze_mock_objects(json_path)
    all_data.extend(result)

df = pd.DataFrame(all_data)
df.to_csv(r'C:\CCTR\result.csv', index=False)

summary = (
    df.groupby("mockPatternLevel")
      .agg({
          "added_cctr": "mean",
          "avg_added_cctr_per_test_case": "mean",
          "test_case_count": "mean"
      })
      .reset_index()
      .sort_values("mockPatternLevel")
)

summary.to_csv(r'C:\CCTR\summary.csv', index=False)

In [13]:
import json

json_path = r"mock object\ActiveMQ.json"
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)


test_case_count = 0
raw_code = set()
for statement in data[0]["statements"]:
    locationContext = statement.get("locationContext", {})
    method_code = locationContext.get("methodRawCode", "")
    if method_code not in raw_code:
        raw_code.add(method_code)
        if statement.get("locate", "")=="Test Case" or "Test" in locationContext.get("methodAnnotations", []) or "test" in locationContext.get("methodName", "").lower():
            test_case_count += 1
if data[0].get("mockPatternLevel",0)==0:
    test_case_count = 1

raw_code = "\n".join(raw_code)
re_mo_code = raw_code
for statement in data[0]["statements"]:
    st_code= statement.get("code", "")
    re_mo_code = re_mo_code.replace(st_code, "")

raw_code_cctr = CognitiveComplexityCalculatorTestAware(raw_code).compute_complexity()
re_mo_code_cctr = CognitiveComplexityCalculatorTestAware(re_mo_code).compute_complexity()
print(raw_code_cctr, re_mo_code_cctr, test_case_count)

18 14 1


In [1]:
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu
import itertools
import numpy as np

# === Step 1: 读取数据 ===
df = pd.read_csv("result.csv")

# 检查列名
required_cols = ["mockPatternLevel", "avg_added_cctr_per_test_case"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"缺少必要列: {col}")

# === Step 2: 按 level 分组数据 ===
groups = {
    level: df[df["mockPatternLevel"] == level]["avg_added_cctr_per_test_case"].dropna()
    for level in sorted(df["mockPatternLevel"].unique())
}


In [ ]:

# === Step 3: Kruskal–Wallis 总体显著性检验 ===
kw_stat, kw_p = kruskal(*groups.values())
print("="*70)
print("Kruskal–Wallis test for CCTR differences among mock levels")
print(f"H statistic = {kw_stat:.3f}, p-value = {kw_p:.5f}")

# === Step 4: 两两比较（Mann–Whitney U + Cliff’s Delta）===
def cliffs_delta(x, y):
    """计算 Cliff's Delta 效应量"""
    m, n = len(x), len(y)
    total = 0
    for xi in x:
        total += np.sum(xi > y) - np.sum(xi < y)
    delta = total / (m * n)
    return delta

print("\nPairwise comparisons (Mann–Whitney U tests):")
for (i, j) in itertools.combinations(groups.keys(), 2):
    g1, g2 = groups[i], groups[j]
    u_stat, p_value = mannwhitneyu(g1, g2, alternative='two-sided')
    delta = cliffs_delta(g1.to_numpy(), g2.to_numpy())
    effect = (
        "negligible" if abs(delta) < 0.147 else
        "small" if abs(delta) < 0.33 else
        "medium" if abs(delta) < 0.474 else
        "large"
    )
    print(f"  Level {i} vs Level {j}: U={u_stat:.2f}, p={p_value:.5f}, "
          f"Cliff’s delta={delta:.3f} ({effect} effect)")

# === Step 5: 解释提示 ===
print("\nInterpretation guide:")
print(" - p < 0.05 → 差异显著")
print(" - Cliff’s delta 绝对值越大表示效应越强：")
print("   negligible < 0.147 < small < 0.33 < medium < 0.474 < large")
print("="*70)
